# Explicabilidade do modelo — RUL FD001

Depois de medir o desempenho e a incerteza, será investigado quais atributos mais contribuem para a previsão. A explicação será global: será observado quanto o erro aumenta quando os valores de um atributo são embaralhados na validação.

Essa importância por permutação indica utilidade preditiva dentro do modelo e dos dados observados. Ela não prova que um sensor causa a degradação nem que o atributo continuará importante em outro conjunto de condições (Fisher; Rudin; Dominici, 2019).

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
import plotly.io as pio
from multilang import LangMap

pio.renderers.default = 'vscode'
lang = LangMap(source='pt', target='pt')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data' / 'raw' / 'dataset').exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cmapss_eda import load_split
from cmapss_modeling import add_rul_label, candidate_rul_features, nonconstant_features
from cmapss_optuna import build_pipeline_from_params, optimize_model
from cmapss_explainability import permutation_importance_by_engine

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'dataset'

## 1. Importância por permutação

Será utilizado o Extra Trees v1, reconstruído a partir do melhor trial. Em cada parte da validação, o modelo será treinado com alguns motores e avaliado em motores que ficaram fora do treinamento. Depois, cada atributo será embaralhado separadamente; se o erro aumentar, será registrado que aquele atributo ajudava a previsão naquele grupo de motores.

In [2]:
train = add_rul_label(load_split(DATA_DIR, 'train', 'FD001'))
features = nonconstant_features(
    train,
    candidate_rul_features(include_settings=True),
)
study = optimize_model(
    train, features=features, model_name='Extra Trees',
    n_trials=100, n_splits=5, random_state=42,
    storage_dir=PROJECT_ROOT / 'outputs' / 'optuna',
    search_version='v1',
)
model = build_pipeline_from_params(
    'Extra Trees', study.best_params, len(features),
    random_state=42, search_version='v1',
)
importance_summary = permutation_importance_by_engine(
    train, features=features, model=model,
    n_splits=5, n_repeats=5, random_state=42,
)
importance_summary.head(15).round(4)

[I 2026-09-28 14:03:44,774] Using an existing study with `study_name='fd001_extra_trees_v1'` instead of creating a new one.


,feature,importance_mean,importance_std_between_folds,repeats_std_mean
0,cycle,15.5992,0.8605,0.3280
1,s11,1.8303,0.4788,0.0901
2,s9,1.1979,0.5398,0.0849
3,s14,0.8553,0.6007,0.0701
4,s12,0.8252,0.4130,0.0506
5,s4,0.7131,0.2758,0.0605
6,s7,0.4801,0.1921,0.0383
7,s15,0.3554,0.1570,0.0283
8,s21,0.2020,0.0887,0.0249
9,s20,0.1699,0.0844,0.0236


### Insights

Os primeiros atributos da tabela foram os que mais aumentaram o erro quando sua informação foi embaralhada. Isso sugere que o modelo dependia mais deles para prever o RUL no FD001. A importância não deve ser confundida com causalidade: um sensor pode ser apenas um indicador associado a outro fenômeno do motor.

In [3]:
top_importance = importance_summary.head(15).sort_values('importance_mean')
fig = px.bar(
    top_importance,
    x='importance_mean',
    y='feature',
    error_x='importance_std_between_folds',
    orientation='h',
    labels={
        'importance_mean': 'Aumento médio do desempenho negativo do MAE',
        'feature': 'Atributo',
    },
    title=lang({'title': 'Atributos mais importantes por permutação — FD001'})['title'],
    height=650,
)
fig.show()

### Insights

As barras de erro mostram quanto a importância variou entre as partes da validação. Uma importância alta e instável deve ser interpretada com cautela, pois pode depender de poucos motores. Uma importância alta e mais consistente será uma candidata melhor para investigação física e para uma futura análise de explicações por motor.

## Referências

FISHER, A.; RUDIN, C.; DOMINICI, F. All Models Are Wrong, but Many Are Useful: Learning a Variable's Importance by Studying an Entire Class of Prediction Models. Journal of Machine Learning Research, v. 20, n. 177, p. 1–81, 2019.